# Fråga 15: Diamantprisprediktion med ML

**Uppgift:** Skapa en ML-modell för diamantprisprediktion och en Streamlit-app.

I detta script demonstrerar vi hur en Machine Learning workflow kan se ut när man använder train, validation och test set.

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

print(" Imports klara!")


 Imports klara!


## Ladda data

In [2]:
# Ladda dataset
df = pd.read_csv('diamonds.csv')
print(f"Dataset: {df.shape[0]:,} rader, {df.shape[1]} kolumner")
print(f"Kolumner: {list(df.columns)}")
print("\nFörsta 3 rader:")
print(df.head(3))

Dataset: 53,940 rader, 11 kolumner
Kolumner: ['Unnamed: 0', 'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y', 'z']

Första 3 rader:
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0  326.0  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0  326.0  3.89  3.84   
2           3   0.23     Good     E     VS1   56.9   65.0  327.0  4.05  4.07   

      z  
0  2.43  
1  2.31  
2  2.31  


## Förbereda data på det sätt som kommer användas

In [3]:
# Rensa data och skapa features
df_clean = df.dropna().copy()  # Ta bort rader med saknade värden och skapa explicit kopia

# Skapa volym-feature
df_clean['volume'] = df_clean['x'] * df_clean['y'] * df_clean['z']

# Välj features och target
features = ['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z', 'volume']
target = 'price'

X = df_clean[features]
y = df_clean[target]

print("X info:")
print(X.info())
print("\ny info:")
print(y.info())

X info:
<class 'pandas.core.frame.DataFrame'>
Index: 53864 entries, 0 to 53917
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53864 non-null  float64
 1   cut      53864 non-null  object 
 2   color    53864 non-null  object 
 3   clarity  53864 non-null  object 
 4   depth    53864 non-null  float64
 5   table    53864 non-null  float64
 6   x        53864 non-null  float64
 7   y        53864 non-null  float64
 8   z        53864 non-null  float64
 9   volume   53864 non-null  float64
dtypes: float64(7), object(3)
memory usage: 4.5+ MB
None

y info:
<class 'pandas.core.series.Series'>
Index: 53864 entries, 0 to 53917
Series name: price
Non-Null Count  Dtype  
--------------  -----  
53864 non-null  float64
dtypes: float64(1)
memory usage: 841.6 KB
None


## Train, Validation och Test Set

In [4]:
# Första uppdelning: train+validation vs test (80% vs 20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Andra uppdelning: train vs validation (70% vs 30% av train_full)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=42)

print(f"Träningsdata: {X_train.shape[0]:,} rader")
print(f"Valideringsdata: {X_val.shape[0]:,} rader")
print(f"Testdata: {X_test.shape[0]:,} rader")
print(f"Features: {X_train.shape[1]} kolumner")

Träningsdata: 30,163 rader
Valideringsdata: 12,928 rader
Testdata: 10,773 rader
Features: 10 kolumner


## EDA

Vi får bara utforska/lära oss saker på träningsdata när vi skapar vår modell.

In [5]:
# Grundläggande statistik på träningsdata
print("Beskrivande statistik (träningsdata):")
print(X_train.describe())

print("\nKategoriska variabler (träningsdata):")
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"{col}: {X_train[col].nunique()} unika värden")

print(f"\nPrisstatistik (träningsdata):")
print(f"Medelpris: ${y_train.mean():,.0f}")
print(f"Minpris: ${y_train.min():,.0f}")
print(f"Maxpris: ${y_train.max():,.0f}")

Beskrivande statistik (träningsdata):
              carat         depth         table             x             y  \
count  30163.000000  30163.000000  30163.000000  30163.000000  30163.000000   
mean       0.796502     61.748692     57.459261      5.727097      5.730034   
std        0.472818      1.429557      2.253650      1.122618      1.124118   
min        0.200000     43.000000     44.000000      0.000000      0.000000   
25%        0.400000     61.000000     56.000000      4.710000      4.720000   
50%        0.700000     61.900000     57.000000      5.690000      5.700000   
75%        1.040000     62.500000     59.000000      6.540000      6.540000   
max        5.010000     79.000000     95.000000     10.740000     31.800000   

                  z        volume  
count  30163.000000  30163.000000  
mean       3.535626    129.529073  
std        0.695477     76.399117  
min        0.000000      0.000000  
25%        2.910000     65.107716  
50%        3.520000    114.585425 

## Pipeline-klass

In [6]:
# Enkel pipeline för diamantprisprediktion
class DiamondPipeline:
    def __init__(self, model=None):
        self.model = model  # Ingen default modell
        self.encoder = OneHotEncoder(drop='first', sparse_output=False)
        self.scaler = StandardScaler()
        
    def fit(self, X, y):
        # Separera numeriska och kategoriska features
        numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z', 'volume']
        categorical_features = ['cut', 'color', 'clarity']
        
        X_num = X[numeric_features]
        X_cat = X[categorical_features]
        
        # Skala numeriska features
        X_num_scaled = self.scaler.fit_transform(X_num)
        
        # Encode kategoriska features till dummy variables
        X_cat_encoded = self.encoder.fit_transform(X_cat)
        
        # Kombinera features
        X_processed = np.hstack([X_num_scaled, X_cat_encoded])
        
        # Träna modellen
        self.model.fit(X_processed, y)
        return self
    
    def predict(self, X):
        # Samma preprocessing som vid träning
        numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z', 'volume']
        categorical_features = ['cut', 'color', 'clarity']
        
        X_num = X[numeric_features]
        X_cat = X[categorical_features]
        
        # Skala numeriska features
        X_num_scaled = self.scaler.transform(X_num)
        
        # Encode kategoriska features
        X_cat_encoded = self.encoder.transform(X_cat)
        
        # Kombinera och prediktera
        X_processed = np.hstack([X_num_scaled, X_cat_encoded])
        return self.model.predict(X_processed)

print(" Pipeline-klass skapad!")

 Pipeline-klass skapad!


## Träna och utvärdera modeller samtidigt

Vi tränar och utvärderar båda modellerna på en gång för effektivitet.

In [7]:
# Träna och utvärdera båda modeller samtidigt
print("Tränar och utvärderar Random Forest och Linear Regression...")

# Random Forest med OPTIMERAD GridSearch (snabb körning)
print("1. Tränar Random Forest med optimerad GridSearch...")


# Definiera preprocessing-steps
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z', 'volume']
categorical_features = ['cut', 'color', 'clarity']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

# Skapa pipeline med preprocessor och Random Forest
rf_pipeline_for_grid = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# OPTIMERADE GridSearch parametrar för snabb körning
rf_param_grid = {
    'regressor__n_estimators': [50, 100],           # Färre träd (snabbare)
    'regressor__max_depth': [10, 20],               # Begränsat djup
    'regressor__min_samples_split': [2, 5],         # Färre split-alternativ
    'regressor__min_samples_leaf': [1, 2]           # Färre leaf-alternativ
}

# Beräkna antal kombinationer
total_combinations = 2 * 2 * 2 * 2
print(f"   GridSearch kommer testa {total_combinations} kombinationer (mycket snabbare!)")

print("   Söker efter bästa hyperparametrar...")
rf_grid_search = GridSearchCV(
    rf_pipeline_for_grid,
    rf_param_grid,
    cv=3,                                          # 3-fold CV (balans mellan tid och kvalitet)
    scoring='neg_mean_squared_error',
    n_jobs=-1,                                     # Använd alla CPU-kärnor
    verbose=0
)

rf_grid_search.fit(X_train, y_train)

print(f"   Bästa parametrar: {rf_grid_search.best_params_}")
print(f"   Bästa CV score: {np.sqrt(-rf_grid_search.best_score_):,.0f}")

# Skapa vår DiamondPipeline med bästa Random Forest
best_rf_params = rf_grid_search.best_params_
best_rf = RandomForestRegressor(
    n_estimators=best_rf_params['regressor__n_estimators'],
    max_depth=best_rf_params['regressor__max_depth'],
    min_samples_split=best_rf_params['regressor__min_samples_split'],
    min_samples_leaf=best_rf_params['regressor__min_samples_leaf'],
    random_state=42
)

rf_pipeline = DiamondPipeline(best_rf)
rf_pipeline.fit(X_train, y_train)

y_val_pred_rf = rf_pipeline.predict(X_val)
rf_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))
print(f"    Random Forest RMSE: ${rf_val_rmse:,.0f}")

# Linear Regression
print("2. Tränar Linear Regression...")
lr_pipeline = DiamondPipeline(LinearRegression())
lr_pipeline.fit(X_train, y_train)

y_val_pred_lr = lr_pipeline.predict(X_val)
lr_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_lr))
print(f"    Linear Regression RMSE: ${lr_val_rmse:,.0f}")

# Jämför och välj bästa modell
print("\nJämförelse av modeller:")
print(f"Random Forest RMSE: ${rf_val_rmse:,.0f}")
print(f"Linear Regression RMSE: ${lr_val_rmse:,.0f}")

if rf_val_rmse < lr_val_rmse:
    print(" Random Forest presterar bättre!")
    best_model = "Random Forest"
    best_pipeline = rf_pipeline
else:
    print(" Linear Regression presterar bättre!")
    best_model = "Linear Regression"
    best_pipeline = lr_pipeline

print(f"Väljer {best_model} som bästa modell.")

Tränar och utvärderar Random Forest och Linear Regression...
1. Tränar Random Forest med optimerad GridSearch...
   GridSearch kommer testa 16 kombinationer (mycket snabbare!)
   Söker efter bästa hyperparametrar...
   Bästa parametrar: {'regressor__max_depth': 20, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}
   Bästa CV score: 717
    Random Forest RMSE: $671
2. Tränar Linear Regression...
    Linear Regression RMSE: $1,330

Jämförelse av modeller:
Random Forest RMSE: $671
Linear Regression RMSE: $1,330
 Random Forest presterar bättre!
Väljer Random Forest som bästa modell.


## Utvärdera vald modell genom testset

Nu tränar vi om vår bästa modell på train + validation data.

In [8]:
# Träna om den bästa modellen på hela träningsdata (train + validation)
if best_model == "Random Forest":
    # Skapa en ny Random Forest med bästa parametrarna från GridSearch
    best_rf_params = rf_grid_search.best_params_
    best_rf = RandomForestRegressor(
        n_estimators=best_rf_params['regressor__n_estimators'],
        max_depth=best_rf_params['regressor__max_depth'],
        min_samples_split=best_rf_params['regressor__min_samples_split'],
        min_samples_leaf=best_rf_params['regressor__min_samples_leaf'],
        random_state=42
    )
    pipeline_final = DiamondPipeline(best_rf)
else:
    # Använd Linear Regression
    pipeline_final = DiamondPipeline(LinearRegression())

pipeline_final.fit(X_train_full, y_train_full)

# Prediktera på testdata
y_test_pred = pipeline_final.predict(X_test)

# Beräkna RMSE på testdata
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"RMSE på testdata: ${test_rmse:,.0f}")
print(f"Genomsnittligt fel: ${test_rmse:,.0f}")

# Visa några exempel
print("\nExempel på testdata:")
for i in range(5):
    print(f"Prediktion: ${y_test_pred[i]:,.0f}, Faktiskt: ${y_test.iloc[i]:,.0f}")

RMSE på testdata: $607
Genomsnittligt fel: $607

Exempel på testdata:
Prediktion: $1,011, Faktiskt: $993
Prediktion: $11,895, Faktiskt: $11,455
Prediktion: $14,791, Faktiskt: $13,953
Prediktion: $5,663, Faktiskt: $4,353
Prediktion: $1,217, Faktiskt: $1,242


## Spara modell för Streamlit-app

In [9]:
# Spara den slutliga modellen
joblib.dump(pipeline_final, 'diamond_pipeline.pkl')
print(" Modell sparad som diamond_pipeline.pkl")

# Testa sparad modell
loaded_pipeline = joblib.load('diamond_pipeline.pkl')
test_sample = X_test.iloc[0:1]
prediction = loaded_pipeline.predict(test_sample)[0]
actual = y_test.iloc[0]
print(f"\nTest: Prediktion ${prediction:,.0f}, Faktiskt ${actual:,.0f}")
print(" Klar för Streamlit!")

 Modell sparad som diamond_pipeline.pkl

Test: Prediktion $1,011, Faktiskt $993
 Klar för Streamlit!
